In [3]:
import os
import sys
import subprocess
import warnings
import pandas as pd
import numpy as np
import torch

warnings.filterwarnings("ignore")

import boto3
import mlflow
import mlflow.pytorch


S3_ENDPOINT = "http://localhost:9000"
AWS_ACCESS_KEY = "admin"
AWS_SECRET_KEY = "password"
BUCKET_NAME = "mlflow-bucket"

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_KEY
os.environ["MLFLOW_S3_ENDPOINT_URL"] = S3_ENDPOINT
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

MLFLOW_TRACKING_URI = "http://localhost:5050"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"   MLflow URI: {mlflow.get_tracking_uri()}")
print(f"   S3 Endpoint: {S3_ENDPOINT}")
print(f"   Bucket: {BUCKET_NAME}")

from mlflow.tracking import MlflowClient
client = MlflowClient()


experiment = client.get_experiment_by_name("nhits_stock_prediction")
prd_runs = []

if experiment:
    runs = client.search_runs(experiment.experiment_id)
    for run in runs:
        tags = run.data.tags
        if tags.get("stage") == "PRD":
            prd_runs.append({
                "run_id": run.info.run_id,
                "name": run.data.tags.get("mlflow.runName", "unknown"),
                "wape": run.data.metrics.get("WAPE", None),
                "mae": run.data.metrics.get("MAE", None)
            })
            print(f"\n   Найдена PRD модель:")
            print(f"   • Run ID: {run.info.run_id}")
            print(f"   • Name: {run.data.tags.get('mlflow.runName', 'unknown')}")
            print(f"   • WAPE: {run.data.metrics.get('WAPE', 'N/A')}")
            print(f"   • MAE: ${run.data.metrics.get('MAE', 'N/A')}")

if not prd_runs:
    print("\n   ⚠️ Модель с тегом PRD не найдена. Использую лучшую из известных.")
    
else:
    # Берем первую PRD модель
    BEST_RUN_ID = prd_runs[0]["run_id"]

print(f"\n Использую Run ID: {BEST_RUN_ID}")



model_uri = f"runs:/{BEST_RUN_ID}/nhits_model"

try:
    model = mlflow.pytorch.load_model(model_uri)
    model.eval()
    print(f"Модель успешно загружена!")

except Exception as e:
    print(f"Ошибка загрузки: {e}")

    
    # Альтернативная загрузка через artifact_uri
    run = client.get_run(BEST_RUN_ID)
    artifact_uri = run.info.artifact_uri
    print(f"   Artifact URI: {artifact_uri}")
    
    model = mlflow.pytorch.load_model(f"{artifact_uri}/nhits_model")
    model.eval()
    print(f"Модель загружена через artifact_uri")


df = pd.read_csv('prices_all.csv')
df['date'] = pd.to_datetime(df['date'])

cluster_df = pd.read_csv('cluster_fullstart_assignments.csv')


run = client.get_run(BEST_RUN_ID)
params = run.data.params

input_size = int(params.get('input_size', 60))
n_clusters = int(params.get('n_clusters', 8))
use_clusters = params.get('use_clusters', 'True') == 'True'


def predict_with_nhits(model, input_prices, cluster_onehot):

    model.eval()
    
    with torch.no_grad():
        # insample_y - исторические цены
        insample_y = torch.from_numpy(input_prices).float().reshape(1, -1, 1)
        
        # insample_mask - маска (все данные валидны)
        insample_mask = torch.ones_like(insample_y)
        
        # stat_exog - статические признаки (кластеры)
        stat_exog = torch.from_numpy(cluster_onehot).float().reshape(1, -1)
        
        # hist_exog - исторические экзогенные признаки
        hist_exog = stat_exog.unsqueeze(1).repeat(1, input_size, 1)
        
        # futr_exog - будущие экзогенные признаки
        horizon = 30
        futr_exog = stat_exog.unsqueeze(1).repeat(1, horizon, 1)
        
        batch = {
            'insample_y': insample_y,
            'insample_mask': insample_mask,
            'stat_exog': stat_exog,
            'hist_exog': hist_exog,
            'futr_exog': futr_exog
        }
        
        output = model(batch)
        predictions = output.squeeze().cpu().numpy()
    
    return predictions

print("ТЕСТОВЫЙ ПРЕДИКТ ДЛЯ AAPL")


aapl_data = df[df['Ticker'] == 'AAPL'].sort_values('date').tail(input_size)
last_prices = aapl_data['Close'].values
aapl_cluster = cluster_df[cluster_df['Company'] == 'AAPL']['Cluster'].values[0]

cluster_onehot = np.zeros(n_clusters)
cluster_onehot[aapl_cluster] = 1

predictions = predict_with_nhits(model, last_prices, cluster_onehot)

print(f"ПРОГНОЗ AAPL НА 30 ДНЕЙ:")

for i, val in enumerate(predictions[:10]):
    print(f"День {i+1:2d}: ${val:.2f}")
print(f"День 30: ${predictions[-1]:.2f}")



print("ПРОГНОЗ ДЛЯ НЕСКОЛЬКИХ ТИКЕРОВ")

sample_tickers = ['AAPL', 'MSFT', 'GOOG', 'AMZN', 'NVDA', 'META', 'TSLA']

results = []
for ticker in sample_tickers:
    ticker_data = df[df['Ticker'] == ticker].sort_values('date').tail(input_size)
    if len(ticker_data) >= input_size:
        prices = ticker_data['Close'].values[-input_size:]
        ticker_cluster = cluster_df[cluster_df['Company'] == ticker]['Cluster'].values
        
        if len(ticker_cluster) > 0:
            cluster_id = ticker_cluster[0]
            cluster_oh = np.zeros(n_clusters)
            cluster_oh[cluster_id] = 1
            
            pred = predict_with_nhits(model, prices, cluster_oh)
            
            results.append({
                'Ticker': ticker,
                'Cluster': cluster_id,
                'Current_Price': prices[-1],
                'Forecast_30d': pred[-1],
                'Change_%': ((pred[-1] - prices[-1]) / prices[-1] * 100),
                'Min_30d': min(pred),
                'Max_30d': max(pred)
            })

# Вывод результатов

print(f"   {'Ticker':<8} {'Cluster':<8} {'Current':<12} {'Forecast':<12} {'Change %':<12} {'Range':<20}")

for r in results:
    print(f"   {r['Ticker']:<8} {r['Cluster']:<8} ${r['Current_Price']:<11.2f} ${r['Forecast_30d']:<11.2f} {r['Change_%']:+.2f}%      ${r['Min_30d']:.2f} - ${r['Max_30d']:.2f}")







   MLflow URI: http://localhost:5050
   S3 Endpoint: http://localhost:9000
   Bucket: mlflow-bucket

   Найдена PRD модель:
   • Run ID: c96e56c31adb4cd9b6ecb26fa5ad58a5
   • Name: cluster_model_v1
   • WAPE: 5.156514413624699
   • MAE: $12.568944814257087

 Использую Run ID: c96e56c31adb4cd9b6ecb26fa5ad58a5


Модель успешно загружена!
ТЕСТОВЫЙ ПРЕДИКТ ДЛЯ AAPL
ПРОГНОЗ AAPL НА 30 ДНЕЙ:
День  1: $277.01
День  2: $280.98
День  3: $283.61
День  4: $283.99
День  5: $290.60
День  6: $290.13
День  7: $290.60
День  8: $296.35
День  9: $303.13
День 10: $305.03
День 30: $385.26
ПРОГНОЗ ДЛЯ НЕСКОЛЬКИХ ТИКЕРОВ
   Ticker   Cluster  Current      Forecast     Change %     Range               
   AAPL     4        $273.67      $385.26      +40.78%      $277.01 - $385.26
   MSFT     4        $485.92      $697.14      +43.47%      $492.22 - $697.14
   GOOG     1        $308.61      $424.60      +37.59%      $311.82 - $424.60
   AMZN     0        $227.35      $328.02      +44.28%      $230.47 - $328.02
   NVDA     3        $180.99      $261.35      +44.40%      $183.42 - $261.35
   META     1        $658.77      $930.65      +41.27%      $666.55 - $930.65
   TSLA     1        $481.20      $672.89      +39.84%      $486.47 - $672.89
